# Level 1 · Part 1 — From raw spatial output to clean cells

*"What did the instrument measure, and which cells can we trust?"*

This section is a **MERSCOPE (MERFISH)** image of **developing human neocortex** — a thin tissue
slice in which a few hundred marker genes were imaged *in situ*, so every detected mRNA molecule
keeps its `(x, y)` position. Unlike dissociated single-cell RNA-seq, we keep **space**: where each
cell sits, who its neighbours are, how the tissue is layered.

The developing cortex has a stereotyped architecture we'll be able to see:

- **Progenitors** (radial glia, intermediate progenitor cells) line the ventricle in the
  **ventricular / subventricular zones (VZ/SVZ)**.
- Newborn neurons **migrate radially** outward and mature **excitatory (glutamatergic) neurons**
  settle into the **cortical plate** in layers (deep layers born first).
- **Inhibitory (GABAergic) interneurons** are born ventrally and migrate in tangentially.
- **Glia** — astrocytes and the oligodendrocyte lineage — and **microglia** and **vascular** cells
  appear and expand later; the **white matter** below the plate is enriched for glia and migrating cells.

Before any of that, though: what exactly did the instrument give us, and how good are the cells the
vendor's pipeline handed back?

Throughout, we drive everything through the **SpatialData** ecosystem — one object holding the image,
the transcripts, the cell polygons and the cell×gene table, all in a shared coordinate system. Keep
the [SpatialData docs & tutorials](https://spatialdata.scverse.org/) open in a tab; they are the
reference for every operation below (Marconato *et al.*, *Nat. Methods* 2025).

## 0. Setup

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import spatialdata as sd
import spatialdata_plot  # noqa: F401  (registers the `.pl` plotting accessor)
from spatialdata import get_extent
from spatialdata.transformations import get_transformation

import warnings

warnings.simplefilter("ignore")  # keep the teaching output readable
sc.settings.verbosity = 1

# The section is staged ONCE, read-only, shared by everyone. We only read it here.
SHARED_ZARR = (
    "/shared/projects/tp_2630_ubordeaux_neuromics_184418/projects/C15/"
    "data/wang2025_merfish/processed/UCSF2018-003-MFG_baseline.zarr"
)

## 1. Load the data and look at its structure

The section is stored as a **SpatialData** object: a multiscale **image** (the stains), a **points**
element (every decoded transcript), a **shapes** element (the vendor's cell polygons), and a
**table** (cells × genes, with per-cell metadata). Printing it shows all four elements and the
single **coordinate system** (`global`) they share.

In [ ]:
sdata = sd.read_zarr(SHARED_ZARR)
sdata

In [ ]:
image_key = list(sdata.images)[0]
points_key = list(sdata.points)[0]
shapes_key = list(sdata.shapes)[0]

full_res = sdata[image_key]["scale0"]["image"]
print(
    "image  :",
    image_key,
    "| shape (C, Y, X) =",
    tuple(full_res.shape),
    "| channels:",
    [str(c) for c in full_res.coords["c"].values],
)
print("points :", points_key, "(individual transcripts)")
print("shapes :", shapes_key, "->", f"{len(sdata[shapes_key]):,}", "vendor cell polygons")
print("table  :", sdata["table"].shape, "(cells x genes)")
sdata["table"].obs.head()

## 2. How is such a huge image stored? The resolution pyramid

The full-resolution image is ~122,000 × 82,000 pixels — about **10 billion** pixels per channel.
You could never hold that in memory just to glance at the slide. SpatialData sidesteps this by
storing the image **multiscale** (a *pyramid*): the **same image at several resolutions**, each level
roughly half the size of the one before (`scale0` = full-res, `scale1` ≈ ½, `scale2` ≈ ¼, …). Levels
are stored **lazily** (as Dask arrays), so pixels are only read from disk when you actually use them.
A whole-slide preview reads a tiny coarse level; a zoom reads full-res *tiles* only where you look.

🔬 **TASK — Explore the pyramid.** For the image element, print each level's `(C, Y, X)` shape and
how many times smaller it is than `scale0`. Then pick the coarsest level that is still ≳ 1000 px wide
— that's a good "whole-slide preview" level.

💡 **HINT.** A multiscale image behaves like a dict of levels: iterate `for level in sdata[image_key]:`
and index `sdata[image_key][level]["image"]` to get the array; compare `.shape[2]` (the X size) across
levels. See *working with multiscale images* in the SpatialData docs.

In [ ]:
# TODO (exercise): print each pyramid level's shape + downsample factor, then choose a
# coarse `preview_level` (>= ~1000 px wide) for whole-slide previews. See the task + hint above.
...

❓ **Question.** The image is stored *multiscale* (a pyramid: `scale0`, `scale1`, …). Why is that
essential for a section that is ~100,000 × 80,000 pixels? *(Hint: what would it cost to draw the
full-resolution image just to preview the whole slide?)*

## 3. The whole section — nuclei and total RNA

Two stains: **DAPI** marks **nuclei**, **poly(T)** marks **total mRNA** (largely cytoplasmic, so it
approximates each cell's full extent). Together they outline where cells are — the raw material any
segmentation works from. We render straight from the SpatialData object with `spatialdata-plot`'s
`.pl` accessor; passing `scale=preview_level` tells it to draw the **coarse pyramid level** from §2
(fast). The tissue is dim with a few bright spots, so we set display contrast from a percentile.

In [ ]:
coarse = sdata[image_key][preview_level]["image"]
dapi_norm = mcolors.Normalize(0, float(np.percentile(np.asarray(coarse.sel(c=0)), 99.5)))
polyt_norm = mcolors.Normalize(0, float(np.percentile(np.asarray(coarse.sel(c=1)), 99.5)))

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
sdata.pl.render_images(image_key, channel=0, cmap="gray", norm=dapi_norm, scale=preview_level, colorbar=False).pl.show(
    ax=axes[0], title="DAPI - nuclei"
)
sdata.pl.render_images(
    image_key, channel=1, cmap="magma", norm=polyt_norm, scale=preview_level, colorbar=False
).pl.show(ax=axes[1], title="poly(T) - total RNA")
fig.tight_layout()

❓ **Question.** Where do you already see structure — a dense band (the cortical plate), sparser
zones, the ventricular edge? The way cells are packed changes across the tissue, and that will make
segmentation easier in some regions than others.

## 4. Zoom in by *cropping* the SpatialData object

To look closely — or later to run heavy methods on a manageable piece — we **crop** to a rectangular
window. In SpatialData a crop is a **bounding-box query**: you give a box in some coordinate system,
and you get back a *new* SpatialData with **every element** (image, transcripts, cell polygons, table)
cut to that box and **kept in sync** with each other. No manual bookkeeping.

Here all elements live in one coordinate system, **`global`**, whose units are **image pixels**.
`get_extent(...)` reports the pixel bounds of an element; the transcripts also carry the
microns→pixels scale, so we read off **1 µm ≈ 9.26 px** in one line and can size a window in microns.

🔬 **TASK — Crop to a ~1 mm window over the cortical plate.** (1) Draw the whole section (coarse
level) with a red `Rectangle` marking the window, so you can *see* where the crop sits. (2) Run the
bounding-box query to produce `crop`, and print it — check that the transcript / cell / table counts
all shrank together.

💡 **HINT.** The query is a method on the object: `sdata.query.bounding_box(axes=("x", "y"),
min_coordinate=[x0, y0], max_coordinate=[x1, y1], target_coordinate_system="global")`. Box corners are
in pixels = microns × `px_per_um`. For the overview rectangle, reuse `render_images(scale=preview_level)`
and add a `matplotlib.patches.Rectangle`. See *querying / cropping* in the SpatialData docs.

In [ ]:
# microns -> pixels, straight from the transcripts' affine (one honest line)
aff = get_transformation(sdata[points_key], "global").to_affine_matrix(input_axes=("x", "y"), output_axes=("x", "y"))
px_per_um = abs(float(aff[0, 0]))

# Window: a 1 mm square over the layered cortical band (centre chosen for you, in pixels).
CX_PX, CY_PX, WIN_UM = 88000, 33000, 1000
half = px_per_um * WIN_UM / 2
cmin = [CX_PX - half, CY_PX - half]
cmax = [CX_PX + half, CY_PX + half]
print(f"1 um = {px_per_um:.2f} px  ->  {WIN_UM} um window = {2 * half:.0f} px")

In [ ]:
# TODO (exercise): (1) draw the whole section with a red Rectangle at [cmin, cmax];
# (2) build `crop` with sdata.query.bounding_box(...). See the task + hint above.
...

❓ **Question.** Compare `crop` to the full `sdata` you printed in §1 — the image, transcripts,
cells and table all shrank. Which single argument to the query guarantees the **table** stays aligned
to the cells that survived the crop, and why does that matter downstream?

## 5. Even closer — boundaries and transcripts

A 1 mm window still holds thousands of cells. Crop **again** (a SpatialData crop returns a SpatialData,
so you can re-crop it) to a ~150 µm close-up, and look at what segmentation actually works from: the
vendor's cell **boundaries** and the raw **transcripts**, both over DAPI. The vendor used the standard
MERSCOPE approach — a **nuclear (DAPI) seed** grown out with a **poly(T) watershed**.

🔬 **TASK — See what segmentation has to work with.** From `crop`, cut a ~150 µm close-up and build two
side-by-side overlays on it: (left) vendor **boundaries** over DAPI, (right) detected **transcripts**
over DAPI.

💡 **HINT.** Same `sdata.query.bounding_box(...)` idiom as §4, on `crop`. Then chain the accessor:
`view.pl.render_images(...)` → `.pl.render_shapes(..., fill_alpha=0, outline_color=...)` (boundaries,
no fill so DAPI shows through) or `.pl.render_points(...)` (transcripts) → `.pl.show(ax=...)`.

In [ ]:
ext = get_extent(crop[image_key], coordinate_system="global")
ccx = 0.5 * (ext["x"][0] + ext["x"][1])
ccy = 0.5 * (ext["y"][0] + ext["y"][1])
hw = px_per_um * 75  # -> ~150 um close-up
view = crop.query.bounding_box(
    axes=("x", "y"),
    min_coordinate=[ccx - hw, ccy - hw],
    max_coordinate=[ccx + hw, ccy + hw],
    target_coordinate_system="global",
)

In [ ]:
# TODO (exercise): build the two overlays on `view` — boundaries over DAPI, transcripts over DAPI.
# See the task + hint above.
...

Each cyan outline is one vendor cell; each grey dot is one detected mRNA. Segmentation decides which
transcripts belong to which cell — and where boundaries touch, that decision is not obvious. **How
well the vendor did this is the subject of Part 2.**

## 6. Who is here? Marker genes in space

The panel targets cell-type markers, so we can already *see* the tissue's composition — no clustering
or annotation yet, just raw signal in space, drawn with `spatialdata-plot`. Two complementary views on
our `crop` (the §3 stains already showed the grand architecture; here we read out *identity*):

- **Cells coloured by expression** (`render_shapes(color=<gene>)`) — colours each vendor cell by a
  marker, so you see *which lineage each cell belongs to*.
- **Individual transcripts** (`render_points(color="gene", groups=[...])`) in the §5 close-up — the raw
  molecules themselves, before any cell assignment.

`render_shapes`/`render_points` read the gene straight from the table's `var_names` / the points'
`gene` column, so we never leave the SpatialData world. For the cell view we colour by a
**log-normalised** layer (raw counts are noisy); we add that layer to the crop's table.

In [ ]:
# add a log-normalised layer to the crop's table for visualisation (keep raw counts too)
ctab = crop["table"]
ctab.layers["counts"] = ctab.X.copy()
tmp = ctab.copy()
sc.pp.normalize_total(tmp)
sc.pp.log1p(tmp)
ctab.layers["lognorm"] = tmp.X

# one representative in-panel marker per broad lineage
candidates = {
    "Excitatory neurons": "SATB2",
    "Radial glia / progenitors": "VIM",
    "Interneurons (GABAergic)": "DLX2",
    "Astrocytes": "AQP4",
    "OPCs (oligodendrocyte lineage)": "PDGFRA",
    "Vascular": "CLDN5",
}
markers = {lin: g for lin, g in candidates.items() if g in ctab.var_names}
list(markers.items())

In [ ]:
# TODO (exercise): 2x3 panel — colour the crop's cells (render_shapes) by each marker in `markers`,
# using the "lognorm" table layer, with a per-gene percentile colour limit. See the section text + docs.
...

❓ **Question.** Does each marker light up a *different* set of cells, or do the same cells look
positive for several lineages at once? This crop sits in the cortical plate, so `SATB2` (excitatory
neurons) should dominate, with scattered `DLX2` interneurons, `AQP4`/`PDGFRA` glia, and rare `CLDN5`
vessels. Cells that appear positive for two mutually-exclusive lineages at once are a warning sign —
exactly the leakage Part 2 measures.

Now the **raw molecules** for three of those markers in the close-up — the actual transcripts the
segmentation must assign. `groups=[...]` picks just those genes out of the 300-gene panel (colour all
300 and the legend explodes).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
(
    view.pl.render_images(image_key, channel=0, cmap="gray", colorbar=False)
    .pl.render_points(
        points_key, color="gene", groups=["SATB2", "AQP4", "CLDN5"], palette=["#ff3b3b", "#26d07c", "#4da6ff"], size=6
    )
    .pl.show(ax=ax, title="Individual transcripts: SATB2 / AQP4 / CLDN5")
)

## 7. Quality control on the vendor cells

Not every segmented object is a good cell. Standard MERSCOPE QC looks at **transcripts per cell**,
**genes per cell**, and **cell volume**, and drops implausible objects. Typical MERSCOPE cells carry
tens–hundreds of transcripts; common cutoffs keep cells above a minimum count / gene / volume, and eye
very high counts with suspicion (often merged **doublets**). We compute these on the full section's
table (raw counts).

🔬 **TASK 7.1 — Compute the QC metrics.** On the table (raw counts in `.X`), compute per-cell
**transcripts** and **genes** detected (volume is already in `.obs`), then look at the summary.
💡 **HINT.** `sc.pp.calculate_qc_metrics` (with `percent_top=None`, `inplace=True`) writes
`total_counts` and `n_genes_by_counts` into `.obs`; then `.obs[[...]].describe()`.

🔬 **TASK 7.2 — Look at the distributions.** Histogram transcripts/cell, genes/cell, and volume.
💡 **HINT.** one `plt.hist` per metric; add a median line so you can see the low shoulder where a cutoff
would naturally sit. (You *decide* the cutoffs in the question below — we don't apply them; these 90,962
cells are already the vendor's post-QC set.)

In [ ]:
# TODO (exercise): compute per-cell QC metrics on raw counts and describe them. See task 7.1.
...

In [ ]:
# TODO (exercise): histogram the three QC metrics with a median line each. See task 7.2.
...

QC is also **spatial**: a detection gradient across the imaging area (e.g. one edge systematically
dimmer) would bias everything downstream. Colour the cells by `total_counts` *in space* — again straight
from SpatialData, no manual scatter.

In [ ]:
# colour every vendor cell by its transcript count; clip at the 99th percentile so a few
# very high-count cells don't flatten the scale. (matplotlib draws each polygon -> ~1 min.)
vmax = float(np.percentile(adata.obs["total_counts"], 99))
fig, ax = plt.subplots(figsize=(9, 7))
(
    sdata.pl.render_shapes(
        shapes_key, color="total_counts", method="matplotlib", cmap="viridis", norm=mcolors.Normalize(0, vmax)
    ).pl.show(ax=ax, title="transcripts / cell, in space")
)

❓ **Question.** From the histograms, roughly where would *you* put the cutoffs for too-few
transcripts, too-few genes, and implausible volume? And from the spatial map — is detection roughly
uniform, or is there a gradient you'd worry about? A subtle failure mode is a **doublet**: one segmented
object that merged two cells, showing up as unusually high counts *and* co-expression of markers from
mutually-exclusive lineages. Hold that thought — leakage across boundaries is exactly what Part 2 measures.

## Summary & what's next

We loaded the section, learned how its huge image is stored (the pyramid), **cropped** it with a
bounding-box query, previewed the tissue's lineages in space, and QC'd the vendor cells — all through
SpatialData. The recurring question — *which transcripts belong to which cell* — is a **segmentation**
question. In **Part 2** we re-segment a window over this same cortical band several ways and measure, concretely, how
much signal each method captures and how clean its cells are.

**Further reading.**
- **SpatialData** (the container you navigated): [docs & tutorials](https://spatialdata.scverse.org/)
  (Marconato *et al.*, *Nat. Methods* 2025) · **Sopa** (the Level-1 segmentation engine):
  [docs](https://gustaveroussy.github.io/sopa/) · **squidpy** (spatial analysis, Level 2):
  [docs](https://squidpy.readthedocs.io/).
- MERSCOPE/MERFISH QC and best practice: Salas *et al.*, *Optimizing Xenium In Situ data utility*
  (*Nat. Methods* 2025, [10.1038/s41592-025-02617-2](https://doi.org/10.1038/s41592-025-02617-2)) — the
  group behind the segmentation-quality metric we use in Part 2.

🚀 **Going further (optional — open-ended).** Pick your own QC thresholds from the distributions above,
apply them to a *copy* of the table, and report how many cells each cutoff removes. Then plot the
**dropped** cells in space (`render_shapes` on a boolean column) — are they scattered randomly, or
concentrated in one region (a tissue edge, a low-detection patch)? What would a spatial cluster of
dropped cells tell you?